In [27]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import copy

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BERT_MODEL = "bert-base-uncased"
print(f"Device: {DEVICE}")

Device: cuda


### Dataset
Returns each modality separately (unlike early fusion which concatenates audio+vision before returning).

In [28]:
class LateFusionDataset(Dataset):
    """
    sample[0] = utterance string
    sample[1] = context string
    sample[2] = np.array (50, 81)   audio
    sample[3] = np.array (50, 371)  vision
    sample[4] = int label (0/1)
    """
    def __init__(self, sample_list, tokenizer, max_length=128):
        self.sample_list = sample_list
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        self.pad_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.sample_list)

    def _encode_text(self, context_text, utterance_text):
        merged_text = (context_text.strip() + " " + utterance_text.strip()).strip()
        merged_ids  = self.tokenizer.encode(merged_text, add_special_tokens=False)
        available   = self.max_length - 2
        if len(merged_ids) > available:
            merged_ids = merged_ids[-available:]          # keep tail (utterance end)
        final_ids  = [self.cls_id] + merged_ids + [self.sep_id]
        final_mask = [1] * len(final_ids)
        pad_len = self.max_length - len(final_ids)
        if pad_len > 0:
            final_ids  += [self.pad_id] * pad_len
            final_mask += [0]           * pad_len
        return (torch.tensor(final_ids,  dtype=torch.long),
                torch.tensor(final_mask, dtype=torch.long))

    def __getitem__(self, idx):
        s = self.sample_list[idx]
        input_ids, attention_mask = self._encode_text(s[1], s[0])
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "audio":          torch.tensor(s[2], dtype=torch.float32),   # (50, 81)
            "vision":         torch.tensor(s[3], dtype=torch.float32),   # (50, 371)
            "label":          torch.tensor(s[4], dtype=torch.float32),
        }

### Model definitions
`SequenceModel` accepts a `cell_type` argument so we can swap RNN ↔ LSTM cleanly.

In [29]:
# ── Text model ────────────────────────────────────────────────────────────────
class TextModel(nn.Module):
    """BERT [CLS] → MLP → 1 logit."""
    def __init__(self, bert_model_name=BERT_MODEL, hidden_dim=256,
                 dropout=0.3, freeze_bert=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False
        bert_hidden = self.bert.config.hidden_size
        # Remove LayerNorm — BERT's CLS output is already well-behaved
        self.classifier = nn.Sequential(
            nn.Linear(bert_hidden, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, input_ids, attention_mask):
        out     = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]       # (B, 768)
        return self.classifier(cls_emb).squeeze(1)     # (B,)


# ── Sequence model (audio or vision) ─────────────────────────────────────────
class SequenceModel(nn.Module):
    """
    (B, T, input_dim) → RNN/LSTM → MLP → 1 logit
    cell_type: 'lstm' | 'rnn'
    """
    def __init__(self, input_dim, hidden_dim=128, mlp_hidden=64,
                 dropout=0.3, bidirectional=True, cell_type="lstm"):
        super().__init__()
        self.bidirectional = bidirectional
        self.cell_type     = cell_type
        self.input_norm    = nn.LayerNorm(input_dim)  # keep this
        rnn_cls = nn.LSTM if cell_type == "lstm" else nn.RNN
        self.rnn = rnn_cls(
            input_size=input_dim, hidden_size=hidden_dim,
            batch_first=True, bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.classifier = nn.Sequential(
            nn.Linear(out_dim, mlp_hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(mlp_hidden, 1),
        )

    def forward(self, x):
        x = self.input_norm(x)
        output = self.rnn(x)
        # LSTM returns (out, (h_n, c_n)); RNN returns (out, h_n)
        h_n = output[1][0] if self.cell_type == "lstm" else output[1]
        if self.bidirectional:
            h = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (B, 2*hidden)
        else:
            h = h_n[-1]                                # (B, hidden)
        return self.classifier(h).squeeze(1)           # (B,)


# ── Combiner ──────────────────────────────────────────────────────────────────
class LateFusionCombiner(nn.Module):
    """
    strategy: 'average' | 'weighted' | 'mlp'
    """
    def __init__(self, strategy="weighted"):
        super().__init__()
        self.strategy = strategy
        if strategy == "weighted":
            self.weights = nn.Parameter(torch.ones(3))
        elif strategy == "mlp":
            self.mlp = nn.Sequential(
                nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1)
            )

    def forward(self, t, a, v):
        if self.strategy == "average":
            return (t + a + v) / 3.0
        elif self.strategy == "weighted":
            w = torch.softmax(self.weights, dim=0)
            return w[0]*t + w[1]*a + w[2]*v
        elif self.strategy == "mlp":
            return self.mlp(torch.stack([t, a, v], dim=1)).squeeze(1)


# ── Full model ────────────────────────────────────────────────────────────────
class LateFusionModel(nn.Module):
    def __init__(self, text_model, audio_model, vision_model, combiner):
        super().__init__()
        self.text_model   = text_model
        self.audio_model  = audio_model
        self.vision_model = vision_model
        self.combiner     = combiner

    def forward(self, input_ids, attention_mask, audio, vision):
        t = self.text_model(input_ids, attention_mask)
        a = self.audio_model(audio)
        v = self.vision_model(vision)
        return self.combiner(t, a, v)

### Training utilities

In [30]:
import pickle
import json

with open("data/sarcasm.pkl", "rb") as f:
    data = pickle.load(f)

with open("data/sarcasm_data.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

def build_samples(split):
    split_data = data[split]

    texts = split_data["text"]
    audios = split_data["audio"]
    visions = split_data["vision"]
    ids = split_data["id"]

    samples = []

    for i in range(len(ids)):
        sample_id = ids[i]

        if isinstance(sample_id, bytes):
            sample_id = sample_id.decode()

        # 取 json 信息
        info = meta[sample_id]

        # 拼接 context
        context_list = info["context"]
        context_str = " ".join(context_list)

        sample = [
            info["utterance"],
            context_str,
            audios[i],     #(50, 81)
            visions[i],    #(50, 371)
            int(info['sarcasm'])
        ]

        samples.append(sample)

    return samples

# 3. build
train_samples = build_samples("train")
valid_samples = build_samples("valid")
train_valid_samples = train_samples + valid_samples
test_samples  = build_samples("test")

In [31]:
def compute_metrics(labels, preds):
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }


def run_epoch(model, loader, criterion, optimizer=None,
              device=DEVICE, max_grad_norm=5.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            audio = batch["audio"].to(device)
            vis   = batch["vision"].to(device)
            lbls  = batch["label"].to(device)

            logits = model(ids, mask, audio, vis)
            loss   = criterion(logits, lbls)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()

            total_loss += loss.item() * lbls.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
            all_labels.extend(lbls.long().cpu().tolist())
            all_preds.extend(preds)

    return total_loss / len(all_labels), compute_metrics(all_labels, all_preds)


def build_loaders(tokenizer, batch_size=16):
    train_ds = LateFusionDataset(train_valid_samples, tokenizer)
    test_ds  = LateFusionDataset(test_samples,        tokenizer)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(test_ds,  batch_size=batch_size, shuffle=False))


def train_and_eval(model, train_loader, test_loader,
                   num_epochs=10, lr_bert=2e-5, lr_seq=1e-3, label=""):
    criterion = nn.BCEWithLogitsLoss()

    # Separate lr for BERT vs the lightweight sequence models + combiner
    optimizer = AdamW([
        {"params": model.text_model.bert.parameters(),       "lr": lr_bert},
        {"params": model.text_model.classifier.parameters(), "lr": lr_seq},
        {"params": model.audio_model.parameters(),           "lr": lr_seq},
        {"params": model.vision_model.parameters(),          "lr": lr_seq},
        {"params": model.combiner.parameters(),              "lr": lr_seq},
    ], weight_decay=1e-2)

    best_f1, best_metrics, best_state = 0.0, {}, None

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, optimizer, max_grad_norm=5.0)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} | "
              f"Train Acc {tr_m['accuracy']:.4f} | "
              f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
        if te_m["f1"] > best_f1:
            best_f1      = te_m["f1"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"  → Best F1: {best_f1:.4f}\n")
    return best_metrics


# Shared tokenizer & data loaders (reused across all experiments)
tokenizer    = AutoTokenizer.from_pretrained(BERT_MODEL)
train_loader, test_loader = build_loaders(tokenizer)

---
### Experiment 1 — Fusion strategy × Training mode

**Motivation:** your early fusion results showed alternating training improved test accuracy by ~3.6% for RNN. We test whether the same holds for late fusion, and which combiner benefits most.

Grid: `{average, weighted, mlp}` × `{joint, sequential}`

- **Joint:** all parameters updated together every step
- **Sequential:** unimodal models trained first (BERT frozen for this phase), then frozen; combiner trained alone

In [32]:
def make_model(fusion_strategy="weighted", cell_type="lstm"):
    """Construct a fresh LateFusionModel."""
    text_m   = TextModel()
    audio_m  = SequenceModel(input_dim=81,  cell_type=cell_type)
    vision_m = SequenceModel(input_dim=371, cell_type=cell_type)
    combiner = LateFusionCombiner(strategy=fusion_strategy)
    return LateFusionModel(text_m, audio_m, vision_m, combiner).to(DEVICE)


def train_sequential(model, train_loader, test_loader,
                     phase1_epochs=5, phase2_epochs=5, lr=2e-5, label=""):
    """
    Sequential training strategy:
      Phase 1 — train unimodal models only (combiner frozen, BERT fine-tuned)
      Phase 2 — freeze unimodal models, train combiner only
    Returns best-F1 test metrics from phase 2.
    """
    criterion = nn.BCEWithLogitsLoss()

    # ── Phase 1: train unimodal branches, freeze combiner ──────────────────
    print(f"  [{label}] Phase 1 — training unimodal models")
    for p in model.combiner.parameters():
        p.requires_grad = False

    opt1 = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-2
    )
    for epoch in range(1, phase1_epochs + 1):
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt1)
        print(f"    P1 Ep {epoch:02d} | Train Acc {tr_m['accuracy']:.4f}")

    # ── Phase 2: freeze unimodal, unfreeze combiner ─────────────────────────
    print(f"  [{label}] Phase 2 — training combiner only")
    for p in model.parameters():
        p.requires_grad = False
    for p in model.combiner.parameters():
        p.requires_grad = True

    combiner_params = list(model.combiner.parameters())
    opt2 = AdamW(combiner_params, lr=1e-3, weight_decay=1e-2) if combiner_params else None
    # opt2 = AdamW(model.combiner.parameters(), lr=1e-3, weight_decay=1e-2)
    best_f1, best_metrics, best_state = 0.0, {}, None

    for epoch in range(1, phase2_epochs + 1):
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt2)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"    P2 Ep {epoch:02d} | "
              f"Train Acc {tr_m['accuracy']:.4f} | "
              f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
        if te_m["f1"] > best_f1:
            best_f1      = te_m["f1"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"  → Best F1: {best_f1:.4f}\n")
    return best_metrics

In [33]:
NUM_EPOCHS   = 10
STRATEGIES   = ["average", "weighted", "mlp"]
results_exp1 = {}   # (strategy, training_mode) -> metrics dict

for strategy in STRATEGIES:
    # Joint training
    key = (strategy, "joint")
    print(f"\n=== {key} ===")
    model = make_model(fusion_strategy=strategy)
    results_exp1[key] = train_and_eval(
        model, train_loader, test_loader,
        num_epochs=NUM_EPOCHS, label=str(key)
    )

    # Sequential training
    key = (strategy, "sequential")
    print(f"\n=== {key} ===")
    model = make_model(fusion_strategy=strategy)
    results_exp1[key] = train_sequential(
        model, train_loader, test_loader,
        phase1_epochs=NUM_EPOCHS//2,
        phase2_epochs=NUM_EPOCHS//2,
        label=str(key)
    )


=== ('average', 'joint') ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [('average', 'joint')] Ep 01 | Train Acc 0.6087 | Test  Acc 0.7246  F1 0.6780
  [('average', 'joint')] Ep 02 | Train Acc 0.7572 | Test  Acc 0.7754  F1 0.7480
  [('average', 'joint')] Ep 03 | Train Acc 0.8931 | Test  Acc 0.7319  F1 0.7132
  [('average', 'joint')] Ep 04 | Train Acc 0.9783 | Test  Acc 0.6884  F1 0.6614
  [('average', 'joint')] Ep 05 | Train Acc 0.9964 | Test  Acc 0.7319  F1 0.6726
  [('average', 'joint')] Ep 06 | Train Acc 0.9964 | Test  Acc 0.7464  F1 0.6789
  [('average', 'joint')] Ep 07 | Train Acc 0.9964 | Test  Acc 0.7029  F1 0.5859
  [('average', 'joint')] Ep 08 | Train Acc 0.9982 | Test  Acc 0.7174  F1 0.6549
  [('average', 'joint')] Ep 09 | Train Acc 1.0000 | Test  Acc 0.6449  F1 0.6797
  [('average', 'joint')] Ep 10 | Train Acc 0.9928 | Test  Acc 0.7319  F1 0.6891
  → Best F1: 0.7480


=== ('average', 'sequential') ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [('mlp', 'joint')] Ep 01 | Train Acc 0.5399 | Test  Acc 0.6739  F1 0.4578
  [('mlp', 'joint')] Ep 02 | Train Acc 0.7083 | Test  Acc 0.6957  F1 0.7200
  [('mlp', 'joint')] Ep 03 | Train Acc 0.8152 | Test  Acc 0.7246  F1 0.6481
  [('mlp', 'joint')] Ep 04 | Train Acc 0.9601 | Test  Acc 0.7174  F1 0.7023
  [('mlp', 'joint')] Ep 05 | Train Acc 0.9783 | Test  Acc 0.6522  F1 0.6842
  [('mlp', 'joint')] Ep 06 | Train Acc 0.9891 | Test  Acc 0.7391  F1 0.7049
  [('mlp', 'joint')] Ep 07 | Train Acc 0.9946 | Test  Acc 0.7246  F1 0.6275
  [('mlp', 'joint')] Ep 08 | Train Acc 0.9964 | Test  Acc 0.7319  F1 0.6891
  [('mlp', 'joint')] Ep 09 | Train Acc 0.9964 | Test  Acc 0.7464  F1 0.7059
  [('mlp', 'joint')] Ep 10 | Train Acc 0.9964 | Test  Acc 0.7681  F1 0.7091
  → Best F1: 0.7200


=== ('mlp', 'sequential') ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [('mlp', 'sequential')] Phase 1 — training unimodal models
    P1 Ep 01 | Train Acc 0.5181
    P1 Ep 02 | Train Acc 0.5181
    P1 Ep 03 | Train Acc 0.5181
    P1 Ep 04 | Train Acc 0.5181
    P1 Ep 05 | Train Acc 0.5181
  [('mlp', 'sequential')] Phase 2 — training combiner only
    P2 Ep 01 | Train Acc 0.5181 | Test  Acc 0.4275  F1 0.5990
    P2 Ep 02 | Train Acc 0.5181 | Test  Acc 0.4275  F1 0.5990
    P2 Ep 03 | Train Acc 0.5181 | Test  Acc 0.4275  F1 0.5990
    P2 Ep 04 | Train Acc 0.6014 | Test  Acc 0.6957  F1 0.6613
    P2 Ep 05 | Train Acc 0.7917 | Test  Acc 0.6957  F1 0.6316
  → Best F1: 0.6613



---
### Experiment 2 — RNN vs LSTM for sequence encoders

**Motivation:** your early fusion results showed BERT+RNN outperformed BERT+LSTM under alternating training (74.64% vs 71.74%). We check if this pattern transfers to the late fusion audio/vision branches.

In [34]:
results_exp2 = {}

for cell_type in ["lstm", "rnn"]:
    key = f"weighted_joint_{cell_type}"
    print(f"\n=== {key} ===")
    model = make_model(fusion_strategy="weighted", cell_type=cell_type)
    results_exp2[key] = train_and_eval(
        model, train_loader, test_loader,
        num_epochs=NUM_EPOCHS, label=key
    )


=== weighted_joint_lstm ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [weighted_joint_lstm] Ep 01 | Train Acc 0.6178 | Test  Acc 0.7101  F1 0.6552
  [weighted_joint_lstm] Ep 02 | Train Acc 0.7736 | Test  Acc 0.7464  F1 0.7009
  [weighted_joint_lstm] Ep 03 | Train Acc 0.9293 | Test  Acc 0.7246  F1 0.5870
  [weighted_joint_lstm] Ep 04 | Train Acc 0.9801 | Test  Acc 0.7319  F1 0.6891
  [weighted_joint_lstm] Ep 05 | Train Acc 0.9909 | Test  Acc 0.7029  F1 0.6822
  [weighted_joint_lstm] Ep 06 | Train Acc 0.9946 | Test  Acc 0.6884  F1 0.6195
  [weighted_joint_lstm] Ep 07 | Train Acc 1.0000 | Test  Acc 0.7174  F1 0.6486
  [weighted_joint_lstm] Ep 08 | Train Acc 0.9982 | Test  Acc 0.7029  F1 0.6306
  [weighted_joint_lstm] Ep 09 | Train Acc 1.0000 | Test  Acc 0.6812  F1 0.6667
  [weighted_joint_lstm] Ep 10 | Train Acc 1.0000 | Test  Acc 0.6957  F1 0.5882
  → Best F1: 0.7009


=== weighted_joint_rnn ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [weighted_joint_rnn] Ep 01 | Train Acc 0.6250 | Test  Acc 0.6377  F1 0.2857
  [weighted_joint_rnn] Ep 02 | Train Acc 0.7808 | Test  Acc 0.6594  F1 0.4337
  [weighted_joint_rnn] Ep 03 | Train Acc 0.9004 | Test  Acc 0.7899  F1 0.7434
  [weighted_joint_rnn] Ep 04 | Train Acc 0.9783 | Test  Acc 0.6884  F1 0.6861
  [weighted_joint_rnn] Ep 05 | Train Acc 0.9891 | Test  Acc 0.7319  F1 0.5843
  [weighted_joint_rnn] Ep 06 | Train Acc 0.9964 | Test  Acc 0.7174  F1 0.6214
  [weighted_joint_rnn] Ep 07 | Train Acc 1.0000 | Test  Acc 0.7174  F1 0.6422
  [weighted_joint_rnn] Ep 08 | Train Acc 1.0000 | Test  Acc 0.7174  F1 0.6355
  [weighted_joint_rnn] Ep 09 | Train Acc 1.0000 | Test  Acc 0.7391  F1 0.6842
  [weighted_joint_rnn] Ep 10 | Train Acc 1.0000 | Test  Acc 0.7029  F1 0.6612
  → Best F1: 0.7434



---
### Experiment 3 — Modality ablation

**Motivation:** your best text-only baseline already hits 73.19%. Ablation will reveal whether audio/vision actually contribute anything, or if BERT is doing all the work.

We zero-out the logit of the excluded modality at inference time by replacing its model with a dummy that always returns 0.

In [35]:
class ZeroModel(nn.Module):
    """Drop-in replacement that always outputs a zero logit — used for ablation."""
    def forward(self, *args, **kwargs):
        # infer batch size from the first argument
        x = args[0]
        return torch.zeros(x.size(0), device=x.device)


def run_ablation(full_model, test_loader, ablate):
    """
    Evaluate full_model with one or more modalities zeroed out.
    ablate: list of modalities to zero, e.g. ['audio', 'vision']
    """
    # Temporarily swap out the ablated branch
    saved = {}
    if "text" in ablate:
        saved["text"]   = full_model.text_model
        full_model.text_model   = ZeroModel().to(DEVICE)
    if "audio" in ablate:
        saved["audio"]  = full_model.audio_model
        full_model.audio_model  = ZeroModel().to(DEVICE)
    if "vision" in ablate:
        saved["vision"] = full_model.vision_model
        full_model.vision_model = ZeroModel().to(DEVICE)

    _, metrics = run_epoch(full_model, test_loader, nn.BCEWithLogitsLoss())

    # Restore
    for k, v in saved.items():
        setattr(full_model, f"{k}_model", v)

    return metrics

In [36]:
# Train a single full model with the best strategy found in Exp 1
# (update fusion_strategy here if Exp 1 reveals a different winner)
print("Training full model for ablation study...")
ablation_model = make_model(fusion_strategy="weighted")
train_and_eval(ablation_model, train_loader, test_loader,
               num_epochs=NUM_EPOCHS, label="ablation_full")

# Run ablation conditions
ablation_conditions = {
    "text only":          ["audio", "vision"],
    "audio only":         ["text",  "vision"],
    "vision only":        ["text",  "audio"],
    "text + audio":       ["vision"],
    "text + vision":      ["audio"],
    "audio + vision":     ["text"],
    "all modalities":     [],
}

results_exp3 = {}
for condition, ablate in ablation_conditions.items():
    results_exp3[condition] = run_ablation(ablation_model, test_loader, ablate)
    print(f"  {condition:20s} | "
          f"Acc {results_exp3[condition]['accuracy']:.4f}  "
          f"F1  {results_exp3[condition]['f1']:.4f}")

Training full model for ablation study...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [ablation_full] Ep 01 | Train Acc 0.6449 | Test  Acc 0.6884  F1 0.6055
  [ablation_full] Ep 02 | Train Acc 0.7880 | Test  Acc 0.6957  F1 0.6379
  [ablation_full] Ep 03 | Train Acc 0.9022 | Test  Acc 0.7174  F1 0.5806
  [ablation_full] Ep 04 | Train Acc 0.9601 | Test  Acc 0.6232  F1 0.2353
  [ablation_full] Ep 05 | Train Acc 0.9801 | Test  Acc 0.7029  F1 0.6555
  [ablation_full] Ep 06 | Train Acc 0.9982 | Test  Acc 0.6884  F1 0.6055
  [ablation_full] Ep 07 | Train Acc 0.9982 | Test  Acc 0.6667  F1 0.5490
  [ablation_full] Ep 08 | Train Acc 0.9982 | Test  Acc 0.6957  F1 0.6111
  [ablation_full] Ep 09 | Train Acc 1.0000 | Test  Acc 0.6812  F1 0.5217
  [ablation_full] Ep 10 | Train Acc 0.9982 | Test  Acc 0.6667  F1 0.5000
  → Best F1: 0.6555

  text only            | Acc 0.7029  F1  0.6612
  audio only           | Acc 0.5580  F1  0.3711
  vision only          | Acc 0.5725  F1  0.4158
  text + audio         | Acc 0.7029  F1  0.6555
  text + vision        | Acc 0.7029  F1  0.6612
  audio +

### Summary Tables

In [37]:
# ── Experiment 1: fusion strategy × training mode ──────────────────────────
rows1 = []
for (strategy, mode), m in results_exp1.items():
    rows1.append({
        "Fusion Strategy": strategy,
        "Training Mode":   mode,
        "Train Acc":       "-",    # not stored; add if needed
        "Test Acc":        f"{m['accuracy']:.4f}",
        "Precision":       f"{m['precision']:.4f}",
        "Recall":          f"{m['recall']:.4f}",
        "F1":              f"{m['f1']:.4f}",
    })
df1 = pd.DataFrame(rows1)
print("=== Exp 1: Fusion Strategy × Training Mode ===")
display(df1)

# ── Experiment 2: RNN vs LSTM ──────────────────────────────────────────────
rows2 = []
for key, m in results_exp2.items():
    rows2.append({
        "Cell Type": key.split("_")[-1].upper(),
        "Test Acc":  f"{m['accuracy']:.4f}",
        "F1":        f"{m['f1']:.4f}",
    })
df2 = pd.DataFrame(rows2)
print("\n=== Exp 2: RNN vs LSTM ===")
display(df2)

# ── Experiment 3: Modality ablation ───────────────────────────────────────
rows3 = []
for condition, m in results_exp3.items():
    rows3.append({
        "Modalities Used": condition,
        "Test Acc":        f"{m['accuracy']:.4f}",
        "Precision":       f"{m['precision']:.4f}",
        "Recall":          f"{m['recall']:.4f}",
        "F1":              f"{m['f1']:.4f}",
    })
df3 = pd.DataFrame(rows3)
print("\n=== Exp 3: Modality Ablation ===")
display(df3)

=== Exp 1: Fusion Strategy × Training Mode ===


,Fusion Strategy,Training Mode,Train Acc,Test Acc,Precision,Recall,F1
0,average,joint,-,0.7754,0.7188,0.7797,0.7480
1,average,sequential,-,0.6087,0.5253,0.8814,0.6582
2,weighted,joint,-,0.7029,0.6047,0.8814,0.7172
3,weighted,sequential,-,0.7609,0.7241,0.7119,0.7179
4,mlp,joint,-,0.6957,0.5934,0.9153,0.7200
5,mlp,sequential,-,0.6957,0.6308,0.6949,0.6613



=== Exp 2: RNN vs LSTM ===


,Cell Type,Test Acc,F1
0,LSTM,0.7464,0.7009
1,RNN,0.7899,0.7434



=== Exp 3: Modality Ablation ===


,Modalities Used,Test Acc,Precision,Recall,F1
0,text only,0.7029,0.6452,0.6780,0.6612
1,audio only,0.5580,0.4737,0.3051,0.3711
2,vision only,0.5725,0.5000,0.3559,0.4158
3,text + audio,0.7029,0.6500,0.6610,0.6555
4,text + vision,0.7029,0.6452,0.6780,0.6612
5,audio + vision,0.5652,0.4872,0.3220,0.3878
6,all modalities,0.7029,0.6500,0.6610,0.6555


In [38]:
w = torch.softmax(ablation_model.combiner.weights, dim=0)
print(f"text={w[0]:.4f}  audio={w[1]:.4f}  vision={w[2]:.4f}")

text=0.3523  audio=0.3265  vision=0.3211


In [39]:
ablation_model.eval()
all_text, all_audio, all_vision = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        audio = batch["audio"].to(DEVICE)
        vis   = batch["vision"].to(DEVICE)

        all_text.append(ablation_model.text_model(ids, mask))
        all_audio.append(ablation_model.audio_model(audio))
        all_vision.append(ablation_model.vision_model(vis))

text_logits   = torch.cat(all_text).cpu()
audio_logits  = torch.cat(all_audio).cpu()
vision_logits = torch.cat(all_vision).cpu()

print(f"Text  — mean: {text_logits.mean():.4f}  std: {text_logits.std():.4f}")
print(f"Audio — mean: {audio_logits.mean():.4f}  std: {audio_logits.std():.4f}")
print(f"Vision— mean: {vision_logits.mean():.4f}  std: {vision_logits.std():.4f}")

Text  — mean: -0.7928  std: 15.8837
Audio — mean: -0.6123  std: 1.0838
Vision— mean: -0.1899  std: 0.3271


In [40]:
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"grad norm: {total_norm:.4f}")  # if consistently >> 1.0, clipping is needed

grad norm: 0.0013
